# 02 · Models as tools

**Workshop:** AI for Actuaries — From Foundations to AI Agents
**Session / Part:** S1.P3–P4  ·  **Slides:** S1.P3.13–22, S1.P4.7–11
**Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), with Satya Sai Mudigonda and Kasyap
**Date:** 24 July 2026 · Four Points by Sheraton, Whitefield, Bangalore
**Model:** `gemini-3.1-flash-lite` (pinned)  ·  **License:** CC BY-NC 4.0

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohanyashraj/ifoa-workshop/blob/main/notebooks/02_models_as_tools.ipynb)

## What this notebook does
Build the tools an agent calls: synthetic ABC Motor data, a leakage trap, a Poisson GLM, an XGBoost challenger, an out-of-time lift table, SHAP on policy 047231, and a fairness spot-check. Runs end-to-end with NO API key.

*All data is hypothetical — ABC Insurer is a fictional entity for teaching only.
The story: Priya Nair (pricing, ABC General) must explain the price of policy
**ABC-MOT-047231** — a 7-year-old SUV, Tier-2, 35% NCB — so her chief actuary
**Arjun Mehta** can sign it.*

In [ ]:
%pip install -q statsmodels xgboost shap scikit-learn

## 1. Generate ABC Motor 2024 (synthetic, seed=42)
Self-contained — no external download.

In [ ]:
import numpy as np, pandas as pd
SEED = 42; rng = np.random.default_rng(SEED)
N = 5000

seg = rng.choice(["Hatchback","Sedan","SUV","MUV"], N, p=[.45,.3,.18,.07])
age = rng.integers(0, 16, N)
ncb = rng.choice([0,25,35,50], N, p=[.35,.25,.25,.15])
region = rng.choice(["Tier1","Tier2","Tier3"], N, p=[.4,.4,.2])
exposure = np.clip(rng.beta(6,2,N), .05, 1.0)

# True frequency (illustrative): base 8.2%, older cars up, NCB down, tier up
base = 0.082
tier_rel = np.select([region=="Tier1",region=="Tier2",region=="Tier3"], [0.95,1.05,1.12])
ncb_rel  = np.select([ncb==0,ncb==25,ncb==35,ncb==50], [1.0,0.90,0.82,0.72])
lam = base * (1 + 0.03*age) * tier_rel * ncb_rel * exposure
claim_count = rng.poisson(lam)
severity = rng.gamma(2.0, 19000, N).round(0)
claim_amount = np.where(claim_count > 0, severity * claim_count, 0.0)

motor = pd.DataFrame(dict(
    policy_id=[f"ABC-MOT-{i:06d}" for i in range(1, N+1)],
    vehicle_segment=seg, vehicle_age_years=age, ncb_pct=ncb, region=region,
    exposure_years=exposure.round(3), claim_count=claim_count,
    claim_amount_inr=claim_amount))
# Make our hero policy real: a 7-yr SUV, Tier-2, 35% NCB
motor.loc[motor.index[0], ["policy_id","vehicle_segment","vehicle_age_years","ncb_pct","region"]] = \
    ["ABC-MOT-047231","SUV",7,35,"Tier2"]
motor["log_exp"] = np.log(motor.exposure_years)   # earned-exposure offset for the GLM
print("Rows:", len(motor), "| frequency:", round(motor.claim_count.sum()/motor.exposure_years.sum(),4))
motor.head()

## 2. The leakage trap — find the column that breaks the model
**TODO:** before running, predict which column gives a *perfect* score, and why.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
motor["claimed"] = (motor.claim_count > 0).astype(int)

leaky = motor[["vehicle_age_years","ncb_pct","claim_amount_inr"]]   # claim_amount is POST-outcome
auc = roc_auc_score(motor.claimed, LogisticRegression(max_iter=500)
                    .fit(leaky, motor.claimed).predict_proba(leaky)[:,1])
print(f"AUC with claim_amount_inr in the features: {auc:.3f}  <-- too good to be true")
print(">>> claim_amount_inr is non-zero exactly when claimed==1. It's LEAKAGE. Drop it.")

## 3. A time-respecting split — fit both models on the same training set

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

FEATURES = ["vehicle_age_years", "ncb_pct", "region", "vehicle_segment"]
tr_idx, te_idx = train_test_split(motor.index, test_size=0.25, random_state=SEED)
X = pd.get_dummies(motor[FEATURES], drop_first=True)
print(f"train {len(tr_idx)}  |  test {len(te_idx)}")

## 4. Tool #1 — the Poisson GLM (fit on train, earned-exposure offset)

In [ ]:
glm_freq = smf.glm(
    "claim_count ~ vehicle_age_years + C(ncb_pct) + C(region)",
    data=motor.loc[tr_idx], offset=motor.log_exp.loc[tr_idx],
    family=sm.families.Poisson()).fit()
print(glm_freq.params.round(3))
print(">>> age +, NCB -, tier + : exactly where your priors sit.")

## 5. Tool #2 — the XGBoost challenger (same training set)

In [ ]:
xgb_freq = XGBRegressor(objective="count:poisson", max_depth=4, learning_rate=0.05,
                        n_estimators=400, random_state=SEED)
xgb_freq.fit(X.loc[tr_idx], motor.claim_count.loc[tr_idx])
print("XGBoost trained on", len(tr_idx), "rows.")

## 6. The honest leaderboard — bias & top-decile lift (test set)

In [ ]:
def wavg(v, w): return np.average(v, weights=w)
actual = motor.claim_count.loc[te_idx]
exp    = motor.exposure_years.loc[te_idx]
pred_glm = glm_freq.predict(motor.loc[te_idx], offset=motor.log_exp.loc[te_idx])
pred_xgb = pd.Series(xgb_freq.predict(X.loc[te_idx]), index=te_idx)

def decile_lift(pred):
    freq = pred / exp                       # rank by predicted FREQUENCY
    d = pd.qcut(freq.rank(method="first"), 10, labels=False)
    g = pd.DataFrame({"a": actual, "e": exp, "d": d}).groupby("d")
    band = g.apply(lambda x: x.a.sum() / x.e.sum(), include_groups=False)
    return band.iloc[-1] / band.iloc[0]

for name, pred in [("GLM", pred_glm), ("XGBoost", pred_xgb)]:
    print(f"{name:8s}  bias {pred.sum()/actual.sum()-1:+.1%}   "
          f"top-decile lift {decile_lift(pred):.1f}x")
print(">>> lift ordering is the robust signal; bias is noisy on a small synthetic")
print("    test set (~80 claims) — exactly why you bias-check on volume, not a handful.")

## 7. Tool #3 — SHAP: additive, per-policy explanation

In [ ]:
import shap
explainer = shap.TreeExplainer(xgb_freq)
sv = explainer(X.loc[te_idx])
idx = 0   # first test policy (a stand-in for ABC-MOT-047231's factor stack)
print("base value      :", round(float(sv.base_values[idx]), 4))
for feat, val in sorted(zip(X.columns, sv.values[idx]), key=lambda t: -abs(t[1]))[:4]:
    print(f"  {feat:22s} {val:+.4f}")
print(">>> base + these signed pushes SUM to the model's output for this policy —")
print("    the additive explanation Arjun's memo reads, factor by factor.")

## 8. Fairness spot-check — calibration within subgroup
Gender is **not** a feature; we audit by it anyway (proxy test).

In [ ]:
rng2 = np.random.default_rng(1)
aud = motor.loc[te_idx].copy()
aud["gender"]   = rng2.choice(["F", "M"], len(aud))
aud["age_band"] = np.where(aud.vehicle_age_years < 5, "new", "old")
aud["pred"]     = pred_xgb.values
rep = (aud.groupby(["gender", "age_band"])[["pred", "claim_count", "exposure_years"]]
          .apply(lambda g: pd.Series({"pred_freq": wavg(g.pred, g.exposure_years),
                                      "obs_freq":  wavg(g.claim_count, g.exposure_years)}))
          .round(3))
print(rep)
print(">>> predicted should track observed closely in each cell; a wide gap => don't ship.")

## TODO — your turn
1. Add `vehicle_segment` interactions to the GLM formula and re-fit.
2. Re-run the fairness check crossing `region` with `age_band`.
*(Solutions in the workshop hub.)*

## Wrap-up
You built the GLM, the XGBoost challenger, the lift referee, SHAP and a fairness audit — **the five tools the capstone agent calls in notebook 05.**

*Demonstrated: models an agent can call, each governed against leakage and bias.*